# 04_analyze_medium_batch.ipynb

## Objetivo

Analizar los resultados del `03_run_medium_batch.ipynb` mediante métricas automáticas para simplificación de texto en español.

## Bloques de métricas

### Bloque 1. Replicación FEINA
- SARI
- BLEU
- Fernández-Huerta
- Compression ratio
- Sentence splits
- Levenshtein similarity
- Exact copies
- Additions proportion
- Deletions proportion

### Bloque 2. Extensión propuesta
- ROUGE
- INFLESZ
- BERTScore

### Bloque 3. Opcional
- SBERT similarity
- evaluación humana (no automatizada en este notebook)

## Lógica metodológica

Este notebook separa la **ejecución** de la **evaluación**.

Primero:
- carga el archivo de resultados del batch mediano
- valida columnas
- calcula métricas fila por fila y por lote

Después:
- resume resultados por modelo
- resume resultados por ruleset
- resume resultados por combinación modelo-ruleset
- exporta tablas listas para análisis posterior y para el TT

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /home/harielpadillasanchez/Documentos/TT/TT2


In [9]:
import pandas as pd
from IPython.display import display
from src.evaluation.metrics import evaluate_dataframe, summarize_metrics
print("Import OK")

Import OK


In [10]:
# from src.evaluation. metrics import (
#     evaluate_row,
#     evaluate_dataframe,
#     summarize_metrics,
# )
# print("Import OK")

In [11]:
from src.evaluation.metrics import evaluate_dataframe, summarize_metrics
print("Import OK")

Import OK


In [12]:
# =========================
# Configuración
# =========================
EXPERIMENT_NAME = "exp_medium_batch_v1_eval"
COMPUTE_BERTSCORE = True
COMPUTE_SBERT = False

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUTS_DIR / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("OUTPUTS_DIR:", OUTPUTS_DIR)
print("METRICS_DIR:", METRICS_DIR)

OUTPUTS_DIR: /home/harielpadillasanchez/Documentos/TT/TT2/outputs
METRICS_DIR: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/metrics


In [13]:
candidate_files = sorted(OUTPUTS_DIR.glob("*medium*analysis*.csv")) + sorted(OUTPUTS_DIR.glob("*medium*.csv"))
candidate_files = [p for p in candidate_files if p.is_file()]

print("Archivos candidatos encontrados:")
for i, p in enumerate(candidate_files):
    print(f"[{i}] {p.name}")

if not candidate_files:
    raise FileNotFoundError("No se encontró ningún CSV de medium batch en outputs/.")

Archivos candidatos encontrados:
[0] exp_medium_batch_v1_20260308_155905_analysis.csv
[1] exp_medium_batch_v1_20260308_155905.csv
[2] exp_medium_batch_v1_20260308_155905_analysis.csv
[3] exp_medium_batch_v1_20260308_155905_batch.csv


In [14]:
selected_idx = 0
INPUT_CSV_PATH = candidate_files[selected_idx]
print("Usando archivo:", INPUT_CSV_PATH)

Usando archivo: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/exp_medium_batch_v1_20260308_155905_analysis.csv


In [15]:
df = pd.read_csv(INPUT_CSV_PATH)
print("Shape:", df.shape)
display(df.head(2))
print(df.columns.tolist())

Shape: (120, 25)


,experiment_id,run_ts,dataset_name,sample_id,model_name,ruleset_name,prompt_type,source_text,reference_text,generated_text,...,output_chars,input_words,output_words,flag_artifacts,flag_prompt_leak,flag_truncation,flag_generic,empty_output,compression_ratio,n_flags
0,exp_medium_batch_v1,20260308_155905,FEINA,5350_LibroBAC.pdf,llama3,R0,zero-shot,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,Yo entiendo que no quieren hacer esto.,...,38,11,7,False,False,False,False,False,0.636364,0
1,exp_medium_batch_v1,20260308_155905,FEINA,5350_LibroBAC.pdf,llama3,R2,zero-shot,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,No quiero que hagan esto.,...,25,11,5,False,False,False,False,False,0.454545,0


['experiment_id', 'run_ts', 'dataset_name', 'sample_id', 'model_name', 'ruleset_name', 'prompt_type', 'source_text', 'reference_text', 'generated_text', 'prompt_text', 'status', 'error_message', 'inference_seconds', 'input_chars', 'output_chars', 'input_words', 'output_words', 'flag_artifacts', 'flag_prompt_leak', 'flag_truncation', 'flag_generic', 'empty_output', 'compression_ratio', 'n_flags']


## Validación de columnas

In [16]:
required_cols = [
    "source_text",
    "reference_text",
    "generated_text",
    "model_name",
    "ruleset_name",
    "sample_id",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas requeridas: {missing}")

print("Columnas requeridas: OK")

Columnas requeridas: OK


In [17]:
work_df = df.copy()

for col in ["source_text", "reference_text", "generated_text"]:
    work_df[col] = (
        work_df[col]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

print("Shape limpio:", work_df.shape)
display(work_df[["sample_id", "model_name", "ruleset_name", "source_text", "reference_text", "generated_text"]].head(2))

Shape limpio: (120, 25)


,sample_id,model_name,ruleset_name,source_text,reference_text,generated_text
0,5350_LibroBAC.pdf,llama3,R0,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,Yo entiendo que no quieren hacer esto.
1,5350_LibroBAC.pdf,llama3,R2,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,No quiero que hagan esto.


## Cálculo de métricas

In [18]:
df_eval = evaluate_dataframe(
    work_df,
    source_col="source_text",
    pred_col="generated_text",
    ref_col="reference_text",
    compute_bertscore=COMPUTE_BERTSCORE,
    compute_sbert=COMPUTE_SBERT,
    bertscore_lang="es",
)

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

In [19]:
print("Shape evaluado:", df_eval.shape)
display(df_eval.head(2))

Shape evaluado: (120, 50)


,experiment_id,run_ts,dataset_name,sample_id,model_name,ruleset_name,prompt_type,source_text,reference_text,generated_text,...,additions_proportion,deletions_proportion,inflesz_pred,inflesz_src,inflesz_delta,rouge1_f,rouge2_f,rougeL_f,bertscore_f1,sbert_similarity
0,exp_medium_batch_v1,20260308_155905,FEINA,5350_LibroBAC.pdf,llama3,R0,zero-shot,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,Yo entiendo que no quieren hacer esto.,...,0.285714,0.545455,93.035,76.898636,16.136364,0.625000,0.142857,0.625000,0.863403,None
1,exp_medium_batch_v1,20260308_155905,FEINA,5350_LibroBAC.pdf,llama3,R2,zero-shot,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,No quiero que hagan esto.,...,0.400000,0.727273,102.155,76.898636,25.256364,0.428571,0.000000,0.285714,0.757454,None


In [20]:
new_metric_cols = [
    "sari",
    "bleu",
    "fernandez_huerta_pred",
    "fernandez_huerta_src",
    "fernandez_huerta_delta",
    "compression_ratio_eval",
    "sentence_splits",
    "levenshtein_similarity",
    "exact_copy",
    "additions_proportion",
    "deletions_proportion",
    "rouge1_f",
    "rouge2_f",
    "rougeL_f",
    "inflesz_pred",
    "inflesz_src",
    "inflesz_delta",
    "bertscore_f1",
    "sbert_similarity",
]

existing_metric_cols = [c for c in new_metric_cols if c in df_eval.columns]
display(df_eval[existing_metric_cols].head(5))

,sari,bleu,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,sentence_splits,levenshtein_similarity,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1,sbert_similarity
0,36.527778,21.573653,89.411429,83.021818,6.389610,0.636364,0,0.420000,0,0.285714,0.545455,0.625000,0.142857,0.625000,93.035000,76.898636,16.136364,0.863403,None
1,30.804612,10.896448,90.440000,83.021818,7.418182,0.454545,0,0.390805,0,0.400000,0.727273,0.428571,0.000000,0.285714,102.155000,76.898636,25.256364,0.757454,None
2,30.696387,7.264340,102.173333,83.021818,19.151515,1.636364,1,0.377358,0,0.833333,0.727273,0.285714,0.076923,0.285714,100.923889,76.898636,24.025253,0.639589,None
3,30.696387,7.264340,102.173333,83.021818,19.151515,1.636364,1,0.364780,0,0.833333,0.727273,0.285714,0.076923,0.285714,100.923889,76.898636,24.025253,0.637837,None
4,34.316906,10.814410,76.781176,67.385455,9.395722,1.030303,0,0.587963,0,0.294118,0.272727,0.517241,0.285714,0.379310,40.905588,32.244091,8.661497,0.783492,None


In [21]:
summary_global = summarize_metrics(df_eval)
display(summary_global)

,sari,bleu,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,sentence_splits,levenshtein_similarity,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
0,34.535024,12.633866,71.762175,74.524894,-2.762719,0.883578,0.525,0.438653,0.0,0.495578,0.591326,0.425408,0.224081,0.380482,58.926195,45.36154,13.564655,0.790496


In [22]:
summary_by_model = summarize_metrics(df_eval, group_cols=["model_name"])
display(summary_by_model.sort_values("sari", ascending=False, na_position="last"))

,model_name,sari,bleu,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,sentence_splits,levenshtein_similarity,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
0,llama3,35.110187,13.615414,72.586515,74.524894,-1.938379,0.698900,0.316667,0.476434,0.0,0.460864,0.618196,0.446691,0.230872,0.395660,61.596256,45.36154,16.234716,0.806120
1,mistral,33.959860,11.652318,70.937834,74.524894,-3.587060,1.068257,0.733333,0.400873,0.0,0.530293,0.564456,0.404126,0.217290,0.365305,56.256134,45.36154,10.894594,0.774872


In [23]:
summary_by_ruleset = summarize_metrics(df_eval, group_cols=["ruleset_name"])
display(summary_by_ruleset.sort_values("sari", ascending=False, na_position="last"))

,ruleset_name,sari,bleu,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,sentence_splits,levenshtein_similarity,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
0,R0,34.766698,13.063290,70.986123,74.524894,-3.538771,0.879637,0.466667,0.433427,0.0,0.483109,0.581649,0.432297,0.234087,0.388530,57.548435,45.36154,12.186895,0.792921
1,R2,34.303350,12.204443,72.538226,74.524894,-1.986668,0.887519,0.583333,0.443880,0.0,0.508048,0.601004,0.418520,0.214075,0.372435,60.303955,45.36154,14.942415,0.788071


In [24]:
summary_by_model_ruleset = summarize_metrics(df_eval, group_cols=["model_name", "ruleset_name"])
display(summary_by_model_ruleset.sort_values(["model_name", "ruleset_name"]))

,model_name,ruleset_name,sari,bleu,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,sentence_splits,levenshtein_similarity,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
0,llama3,R0,35.325520,14.236782,71.691759,74.524894,-2.833135,0.703941,0.266667,0.478975,0.0,0.433321,0.597364,0.459921,0.243639,0.410400,59.741372,45.36154,14.379832,0.809943
1,llama3,R2,34.894855,12.994047,73.481271,74.524894,-1.043622,0.693858,0.366667,0.473893,0.0,0.488406,0.639028,0.433461,0.218106,0.380920,63.451140,45.36154,18.089600,0.802298
2,mistral,R0,34.207876,11.889797,70.280487,74.524894,-4.244407,1.055333,0.666667,0.387879,0.0,0.532896,0.565933,0.404673,0.224535,0.366660,55.355498,45.36154,9.993958,0.775899
3,mistral,R2,33.711844,11.414838,71.595181,74.524894,-2.929713,1.081180,0.800000,0.413867,0.0,0.527690,0.562979,0.403579,0.210045,0.363949,57.156770,45.36154,11.795230,0.773845


In [25]:
bloque_1 = [
    "sari", "bleu", "fernandez_huerta_pred", "fernandez_huerta_src",
    "fernandez_huerta_delta", "compression_ratio_eval", "sentence_splits",
    "levenshtein_similarity", "exact_copy", "additions_proportion",
    "deletions_proportion",
]

bloque_2 = [
    "rouge1_f", "rouge2_f", "rougeL_f", "inflesz_pred",
    "inflesz_src", "inflesz_delta", "bertscore_f1",
]

bloque_3 = ["sbert_similarity"]

print("Bloque 1 - Replicación FEINA")
display(summary_by_model_ruleset[[c for c in ["model_name", "ruleset_name"] + bloque_1 if c in summary_by_model_ruleset.columns]])

print("Bloque 2 - Extensión propuesta")
display(summary_by_model_ruleset[[c for c in ["model_name", "ruleset_name"] + bloque_2 if c in summary_by_model_ruleset.columns]])

print("Bloque 3 - Opcional")
display(summary_by_model_ruleset[[c for c in ["model_name", "ruleset_name"] + bloque_3 if c in summary_by_model_ruleset.columns]])

Bloque 1 - Replicación FEINA


,model_name,ruleset_name,sari,bleu,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,sentence_splits,levenshtein_similarity,exact_copy,additions_proportion,deletions_proportion
0,llama3,R0,35.325520,14.236782,71.691759,74.524894,-2.833135,0.703941,0.266667,0.478975,0.0,0.433321,0.597364
1,llama3,R2,34.894855,12.994047,73.481271,74.524894,-1.043622,0.693858,0.366667,0.473893,0.0,0.488406,0.639028
2,mistral,R0,34.207876,11.889797,70.280487,74.524894,-4.244407,1.055333,0.666667,0.387879,0.0,0.532896,0.565933
3,mistral,R2,33.711844,11.414838,71.595181,74.524894,-2.929713,1.081180,0.800000,0.413867,0.0,0.527690,0.562979


Bloque 2 - Extensión propuesta


,model_name,ruleset_name,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
0,llama3,R0,0.459921,0.243639,0.410400,59.741372,45.36154,14.379832,0.809943
1,llama3,R2,0.433461,0.218106,0.380920,63.451140,45.36154,18.089600,0.802298
2,mistral,R0,0.404673,0.224535,0.366660,55.355498,45.36154,9.993958,0.775899
3,mistral,R2,0.403579,0.210045,0.363949,57.156770,45.36154,11.795230,0.773845


Bloque 3 - Opcional


,model_name,ruleset_name
0,llama3,R0
1,llama3,R2
2,mistral,R0
3,mistral,R2


In [26]:
display_cols = [
    "sample_id", "model_name", "ruleset_name", "source_text",
    "reference_text", "generated_text", "sari", "bleu", "rougeL_f",
    "bertscore_f1", "compression_ratio_eval", "levenshtein_similarity",
    "exact_copy",
]

display(df_eval[[c for c in display_cols if c in df_eval.columns]].head(10))

,sample_id,model_name,ruleset_name,source_text,reference_text,generated_text,sari,bleu,rougeL_f,bertscore_f1,compression_ratio_eval,levenshtein_similarity,exact_copy
0,5350_LibroBAC.pdf,llama3,R0,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,Yo entiendo que no quieren hacer esto.,36.527778,21.573653,0.625000,0.863403,0.636364,0.420000,0
1,5350_LibroBAC.pdf,llama3,R2,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,No quiero que hagan esto.,30.804612,10.896448,0.285714,0.757454,0.454545,0.390805,0
2,5350_LibroBAC.pdf,mistral,R0,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,Esto quizás no les guste mucho hacer esto. (Th...,30.696387,7.264340,0.285714,0.639589,1.636364,0.377358,0
3,5350_LibroBAC.pdf,mistral,R2,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,Esto quizás no les guste mucho hacer esto. (Th...,30.696387,7.264340,0.285714,0.637837,1.636364,0.364780,0
4,418_LibroUAC_Sincopyright.pdf,llama3,R0,Lo calcula el Instituto Nacional de Estadístic...,Lo calcula el Instituto Nacional de Estadístic...,El Instituto Nacional de Estadísticas calcula ...,34.316906,10.814410,0.379310,0.783492,1.030303,0.587963,0
5,418_LibroUAC_Sincopyright.pdf,llama3,R2,Lo calcula el Instituto Nacional de Estadístic...,Lo calcula el Instituto Nacional de Estadístic...,El Instituto Nacional de Estadísticas calcula ...,44.512493,12.585217,0.400000,0.792587,0.787879,0.530504,0
6,418_LibroUAC_Sincopyright.pdf,mistral,R0,Lo calcula el Instituto Nacional de Estadístic...,Lo calcula el Instituto Nacional de Estadístic...,El Instituto Nacional de Estadísticas calcula ...,34.058874,9.943313,0.377358,0.783019,0.848485,0.794189,0
7,418_LibroUAC_Sincopyright.pdf,mistral,R2,Lo calcula el Instituto Nacional de Estadístic...,Lo calcula el Instituto Nacional de Estadístic...,El Instituto Nacional de Estadísticas calcula ...,35.379081,10.858944,0.415094,0.799675,0.848485,0.790361,0
8,16403_LibroBAC.pdf,llama3,R0,Fijar el precio de los activos en el momento d...,Fijar el precio de los recursos en su emisión ...,Establecer el precio de los activos al venderl...,35.177856,11.408425,0.322581,0.796406,0.857143,0.669421,0
9,16403_LibroBAC.pdf,llama3,R2,Fijar el precio de los activos en el momento d...,Fijar el precio de los recursos en su emisión ...,Establecer el precio de los activos cuando se ...,51.899581,21.401603,0.400000,0.867477,0.571429,0.412698,0


In [27]:
if "sari" in df_eval.columns:
    best_sari = df_eval.sort_values("sari", ascending=False, na_position="last")
    display(best_sari[[c for c in display_cols if c in best_sari.columns]].head(10))

,sample_id,model_name,ruleset_name,source_text,reference_text,generated_text,sari,bleu,rougeL_f,bertscore_f1,compression_ratio_eval,levenshtein_similarity,exact_copy
69,1267_LibroUide_Sincopyright.pdf,llama3,R2,De esta forma se comprueba que el estado finan...,Así comprobamos que el estado financiero está ...,Se verifica que el estado financiero está bien...,75.359564,53.728497,0.666667,0.883180,0.692308,0.577465,0
116,2238_LibroUide_Sincopyright.pdf,llama3,R0,Se iniciará con la explicación de la tabla de ...,Explicaremos la tabla de datos para una variab...,Comenzaremos explicando cómo leer una tabla de...,66.822826,44.749354,0.514286,0.845722,0.750000,0.560311,0
68,1267_LibroUide_Sincopyright.pdf,llama3,R0,De esta forma se comprueba que el estado finan...,Así comprobamos que el estado financiero está ...,Verifica que el estado financiero esté bien he...,65.873155,29.558013,0.705882,0.856998,0.615385,0.561151,0
20,754_LibroUAC_Sincopyright.pdf,llama3,R0,"El Código Tributario, por su parte, señala, en...",El Código Tributario en el artículo diecisiete...,Los contribuyentes deben conservar sus libros ...,55.687546,17.905600,0.490566,0.833593,0.425532,0.481562,0
21,754_LibroUAC_Sincopyright.pdf,llama3,R2,"El Código Tributario, por su parte, señala, en...",El Código Tributario en el artículo diecisiete...,Los contribuyentes deben conservar sus libros ...,55.687546,17.905600,0.490566,0.833593,0.425532,0.481562,0
9,16403_LibroBAC.pdf,llama3,R2,Fijar el precio de los activos en el momento d...,Fijar el precio de los recursos en su emisión ...,Establecer el precio de los activos cuando se ...,51.899581,21.401603,0.400000,0.867477,0.571429,0.412698,0
46,837_LibroUide_Sincopyright.pdf,mistral,R0,A través del desarrollo de los once capítulos ...,En los once capítulos teóricos y prácticos de ...,"En este documento de once capítulos, teoría y ...",51.812174,31.754124,0.591549,0.860060,0.765432,0.063525,0
35,5775_LibroBAC.pdf,mistral,R2,Desde luego en este grupo puede haber categorí...,"Dentro de este grupo hay categorías, clases o ...","En este grupo hay posiblemente categorías, cla...",51.633331,14.100025,0.466667,0.749110,1.727273,0.489362,0
62,583_LibroNEFE_Sincopyright.pdf,mistral,R0,Cuando depositas dinero en una cuenta de ahorr...,El monto ahorrado cuando depositas dinero en u...,Cuando depositas dinero en una cuenta de ahorr...,50.957176,31.592889,0.562500,0.849317,0.833333,0.729167,0
63,583_LibroNEFE_Sincopyright.pdf,mistral,R2,Cuando depositas dinero en una cuenta de ahorr...,El monto ahorrado cuando depositas dinero en u...,Cuando depositas dinero en una cuenta de ahorr...,50.957176,31.592889,0.562500,0.849317,0.833333,0.729167,0


In [28]:
if "sari" in df_eval.columns:
    worst_sari = df_eval.sort_values("sari", ascending=True, na_position="last")
    display(worst_sari[[c for c in display_cols if c in worst_sari.columns]].head(10))

,sample_id,model_name,ruleset_name,source_text,reference_text,generated_text,sari,bleu,rougeL_f,bertscore_f1,compression_ratio_eval,levenshtein_similarity,exact_copy
14,17252_LibroBAC.pdf,mistral,R0,Desviando su correspondencia hacia otra ubicac...,Desviar su correspondencia hacia otra ubicació...,Usando un formulario para redirigir tu correo ...,9.748932,6.315134,0.166667,0.771092,0.769231,0.386667,0
15,17252_LibroBAC.pdf,mistral,R2,Desviando su correspondencia hacia otra ubicac...,Desviar su correspondencia hacia otra ubicació...,Usando un formulario para cambiar la dirección...,14.357448,9.238430,0.275862,0.806721,1.000000,0.463277,0
95,14081_LibroBAC.pdf,mistral,R2,"Por todo lo anterior, este capítulo plantea el...","Por todo lo anterior, este capítulo plantea el...",Este capítulo explica concepto y objetivos de ...,15.509726,4.412865,0.367347,0.836585,0.642857,0.603390,0
38,17825_LibroBAC.pdf,mistral,R0,Sistema económico: Forma de organización socia...,"Sistema económico, forma de organización socia...",Sistema económico: Forma de solucionar problem...,16.841092,3.701296,0.400000,0.794631,0.555556,0.547264,0
92,14081_LibroBAC.pdf,llama3,R0,"Por todo lo anterior, este capítulo plantea el...","Por todo lo anterior, este capítulo plantea el...",Este capítulo explica qué son los impuestos y ...,17.607322,3.890762,0.250000,0.760589,0.607143,0.362264,0
93,14081_LibroBAC.pdf,llama3,R2,"Por todo lo anterior, este capítulo plantea el...","Por todo lo anterior, este capítulo plantea el...",Este capítulo explica qué son los impuestos y ...,17.607322,3.890762,0.250000,0.760245,0.607143,0.356061,0
37,17825_LibroBAC.pdf,llama3,R2,Sistema económico: Forma de organización socia...,"Sistema económico, forma de organización socia...",Sistema económico: Cómo un país organiza su ec...,18.052674,3.676612,0.238095,0.786993,0.611111,0.585859,0
94,14081_LibroBAC.pdf,mistral,R0,"Por todo lo anterior, este capítulo plantea el...","Por todo lo anterior, este capítulo plantea el...",Este capítulo explica concepto y objetivos de ...,18.153974,4.253635,0.468085,0.811132,0.535714,0.354610,0
39,17825_LibroBAC.pdf,mistral,R2,Sistema económico: Forma de organización socia...,"Sistema económico, forma de organización socia...",Sistema económico: Forma de solucionar problem...,18.318943,4.838693,0.450000,0.776790,0.555556,0.578680,0
13,17252_LibroBAC.pdf,llama3,R2,Desviando su correspondencia hacia otra ubicac...,Desviar su correspondencia hacia otra ubicació...,Cambie su dirección en el formulario de cambio...,19.073052,19.767438,0.461538,0.815849,0.769231,0.596026,0


In [29]:
if "exact_copy" in df_eval.columns:
    exact_copies_df = df_eval[df_eval["exact_copy"] == 1].copy()
    print("Exact copies:", len(exact_copies_df))
    display(exact_copies_df[[c for c in display_cols if c in exact_copies_df.columns]].head(10))

Exact copies: 0


,sample_id,model_name,ruleset_name,source_text,reference_text,generated_text,sari,bleu,rougeL_f,bertscore_f1,compression_ratio_eval,levenshtein_similarity,exact_copy


In [30]:
if "additions_proportion" in df_eval.columns:
    high_add = df_eval.sort_values("additions_proportion", ascending=False, na_position="last")
    display(high_add[[c for c in display_cols + ["additions_proportion"] if c in high_add.columns]].head(10))

,sample_id,model_name,ruleset_name,source_text,reference_text,generated_text,sari,bleu,rougeL_f,bertscore_f1,compression_ratio_eval,levenshtein_similarity,exact_copy,additions_proportion
53,1194_LibroNEFE_Sincopyright.pdf,llama3,R2,"Sin perder el ánimo, el mes siguiente retomaro...",El mes siguiente retomaron su plan de ahorros.,No se rindieron y volvieron a ahorrar.,21.036220,4.873499,0.000000,0.734426,0.583333,0.419048,0,1.000000
26,2_LibroBAC.pdf,mistral,R0,"Pero, algo más grave aún, o quizás por eso mis...","Pero, algo más grave aún es que las personas a...",Pero una cosa still more serious is that peopl...,22.696609,0.946055,0.036697,0.768848,0.843750,0.026918,0,0.981481
27,2_LibroBAC.pdf,mistral,R2,"Pero, algo más grave aún, o quizás por eso mis...","Pero, algo más grave aún es que las personas a...","Pero una cosa más grave still, or perhaps for ...",23.780158,1.842643,0.076190,0.750709,0.765625,0.037249,0,0.938776
79,102_LibroNEFE_Sincopyright.pdf,mistral,R2,'Mis padres no se lo podían permitir'.,"""Mis padres no se lo podían permitir"".",Mi familia no pudo permitirlo. (Literally: My ...,25.138889,2.908318,0.100000,0.671969,1.714286,0.357143,0,0.916667
91,11507_LibroBAC.pdf,mistral,R2,La importancia de un historial de crédito impe...,La importancia de un historial crediticio limp...,Importante haber un buen historial de crédito....,36.381992,6.668493,0.212121,0.627879,2.833333,0.123515,0,0.862745
36,17825_LibroBAC.pdf,llama3,R0,Sistema económico: Forma de organización socia...,"Sistema económico, forma de organización socia...",Un sistema económico es cómo un país organiza ...,19.980749,2.168845,0.260870,0.794773,0.777778,0.616822,0,0.857143
3,5350_LibroBAC.pdf,mistral,R2,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,Esto quizás no les guste mucho hacer esto. (Th...,30.696387,7.264340,0.285714,0.637837,1.636364,0.364780,0,0.833333
2,5350_LibroBAC.pdf,mistral,R0,Yo sé que esto es algo que probablemente prefe...,Yo sé que probablemente preferirían no hacer e...,Esto quizás no les guste mucho hacer esto. (Th...,30.696387,7.264340,0.285714,0.639589,1.636364,0.377358,0,0.833333
37,17825_LibroBAC.pdf,llama3,R2,Sistema económico: Forma de organización socia...,"Sistema económico, forma de organización socia...",Sistema económico: Cómo un país organiza su ec...,18.052674,3.676612,0.238095,0.786993,0.611111,0.585859,0,0.818182
70,1267_LibroUide_Sincopyright.pdf,mistral,R0,De esta forma se comprueba que el estado finan...,Así comprobamos que el estado financiero está ...,Se verifica que los libros contables estén bie...,33.563391,4.932352,0.300000,0.803025,0.769231,0.425000,0,0.800000


In [31]:
if "deletions_proportion" in df_eval.columns:
    high_del = df_eval.sort_values("deletions_proportion", ascending=False, na_position="last")
    display(high_del[[c for c in display_cols + ["deletions_proportion"] if c in high_del.columns]].head(10))

,sample_id,model_name,ruleset_name,source_text,reference_text,generated_text,sari,bleu,rougeL_f,bertscore_f1,compression_ratio_eval,levenshtein_similarity,exact_copy,deletions_proportion
53,1194_LibroNEFE_Sincopyright.pdf,llama3,R2,"Sin perder el ánimo, el mes siguiente retomaro...",El mes siguiente retomaron su plan de ahorros.,No se rindieron y volvieron a ahorrar.,21.036220,4.873499,0.000000,0.734426,0.583333,0.419048,0,1.000000
26,2_LibroBAC.pdf,mistral,R0,"Pero, algo más grave aún, o quizás por eso mis...","Pero, algo más grave aún es que las personas a...",Pero una cosa still more serious is that peopl...,22.696609,0.946055,0.036697,0.768848,0.843750,0.026918,0,0.984375
27,2_LibroBAC.pdf,mistral,R2,"Pero, algo más grave aún, o quizás por eso mis...","Pero, algo más grave aún es que las personas a...","Pero una cosa más grave still, or perhaps for ...",23.780158,1.842643,0.076190,0.750709,0.765625,0.037249,0,0.953125
36,17825_LibroBAC.pdf,llama3,R0,Sistema económico: Forma de organización socia...,"Sistema económico, forma de organización socia...",Un sistema económico es cómo un país organiza ...,19.980749,2.168845,0.260870,0.794773,0.777778,0.616822,0,0.888889
37,17825_LibroBAC.pdf,llama3,R2,Sistema económico: Forma de organización socia...,"Sistema económico, forma de organización socia...",Sistema económico: Cómo un país organiza su ec...,18.052674,3.676612,0.238095,0.786993,0.611111,0.585859,0,0.888889
79,102_LibroNEFE_Sincopyright.pdf,mistral,R2,'Mis padres no se lo podían permitir'.,"""Mis padres no se lo podían permitir"".",Mi familia no pudo permitirlo. (Literally: My ...,25.138889,2.908318,0.100000,0.671969,1.714286,0.357143,0,0.857143
14,17252_LibroBAC.pdf,mistral,R0,Desviando su correspondencia hacia otra ubicac...,Desviar su correspondencia hacia otra ubicació...,Usando un formulario para redirigir tu correo ...,9.748932,6.315134,0.166667,0.771092,0.769231,0.386667,0,0.846154
71,1267_LibroUide_Sincopyright.pdf,mistral,R2,De esta forma se comprueba que el estado finan...,Así comprobamos que el estado financiero está ...,Se verifica que los libros estén bien hechos f...,33.689653,5.522398,0.315789,0.803506,0.692308,0.453333,0,0.846154
70,1267_LibroUide_Sincopyright.pdf,mistral,R0,De esta forma se comprueba que el estado finan...,Así comprobamos que el estado financiero está ...,Se verifica que los libros contables estén bie...,33.563391,4.932352,0.300000,0.803025,0.769231,0.425000,0,0.846154
113,3995_LibroBAC.pdf,llama3,R2,"Se demuestra entonces, con este ejemplo, que l...",Entonces se demuestra que la productividad aum...,La productividad es clave para crear más emple...,30.063406,3.136424,0.148148,0.781756,0.440000,0.341014,0,0.840000


In [32]:
pivot_sari = summary_by_model_ruleset.pivot(index="model_name", columns="ruleset_name", values="sari")
pivot_bleu = summary_by_model_ruleset.pivot(index="model_name", columns="ruleset_name", values="bleu")
pivot_rougeL = summary_by_model_ruleset.pivot(index="model_name", columns="ruleset_name", values="rougeL_f")
pivot_bert = summary_by_model_ruleset.pivot(index="model_name", columns="ruleset_name", values="bertscore_f1")
pivot_comp = summary_by_model_ruleset.pivot(index="model_name", columns="ruleset_name", values="compression_ratio_eval")

print("SARI")
display(pivot_sari)

print("BLEU")
display(pivot_bleu)

print("ROUGE-L")
display(pivot_rougeL)

print("BERTScore")
display(pivot_bert)

print("Compression ratio")
display(pivot_comp)

SARI


ruleset_name,R0,R2
model_name,,
llama3,35.325520,34.894855
mistral,34.207876,33.711844


BLEU


ruleset_name,R0,R2
model_name,,
llama3,14.236782,12.994047
mistral,11.889797,11.414838


ROUGE-L


ruleset_name,R0,R2
model_name,,
llama3,0.41040,0.380920
mistral,0.36666,0.363949


BERTScore


ruleset_name,R0,R2
model_name,,
llama3,0.809943,0.802298
mistral,0.775899,0.773845


Compression ratio


ruleset_name,R0,R2
model_name,,
llama3,0.703941,0.693858
mistral,1.055333,1.081180


In [33]:
base_name = INPUT_CSV_PATH.stem.replace("_analysis", "")

EVAL_CSV_PATH = METRICS_DIR / f"{base_name}_evaluated.csv"
SUMMARY_MODEL_PATH = METRICS_DIR / f"{base_name}_summary_by_model.csv"
SUMMARY_RULESET_PATH = METRICS_DIR / f"{base_name}_summary_by_ruleset.csv"
SUMMARY_MODEL_RULESET_PATH = METRICS_DIR / f"{base_name}_summary_by_model_ruleset.csv"

df_eval.to_csv(EVAL_CSV_PATH, index=False, encoding="utf-8")
summary_by_model.to_csv(SUMMARY_MODEL_PATH, index=False, encoding="utf-8")
summary_by_ruleset.to_csv(SUMMARY_RULESET_PATH, index=False, encoding="utf-8")
summary_by_model_ruleset.to_csv(SUMMARY_MODEL_RULESET_PATH, index=False, encoding="utf-8")

print("EVAL_CSV_PATH:", EVAL_CSV_PATH)
print("SUMMARY_MODEL_PATH:", SUMMARY_MODEL_PATH)
print("SUMMARY_RULESET_PATH:", SUMMARY_RULESET_PATH)
print("SUMMARY_MODEL_RULESET_PATH:", SUMMARY_MODEL_RULESET_PATH)

EVAL_CSV_PATH: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/metrics/exp_medium_batch_v1_20260308_155905_evaluated.csv
SUMMARY_MODEL_PATH: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/metrics/exp_medium_batch_v1_20260308_155905_summary_by_model.csv
SUMMARY_RULESET_PATH: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/metrics/exp_medium_batch_v1_20260308_155905_summary_by_ruleset.csv
SUMMARY_MODEL_RULESET_PATH: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/metrics/exp_medium_batch_v1_20260308_155905_summary_by_model_ruleset.csv


### Conclusiones de la corrida experimental (batch mediano)

Después de ejecutar el pipeline con el conjunto de textos del batch mediano y evaluar las salidas generadas por los modelos, se pueden observar varios resultados importantes.

Primero, al comparar los modelos utilizados, Llama 3 mostró un mejor desempeño general que Mistral. En la mayoría de las métricas evaluadas, como SARI, BLEU, ROUGE-L y BERTScore, Llama 3 obtuvo valores ligeramente más altos. Esto indica que sus simplificaciones lograron conservar mejor el significado del texto original mientras realizaban modificaciones para hacerlo más sencillo.

Además, se observó que Llama 3 tiende a generar textos más compactos, es decir, versiones simplificadas un poco más cortas que el texto original. En cambio, Mistral en varios casos mantuvo o incluso aumentó la longitud del texto, lo cual no siempre es deseable cuando se busca simplificar la información.

En cuanto a los conjuntos de reglas utilizados en los prompts, los resultados muestran una diferencia pequeña entre ellos. El ruleset R0 obtuvo resultados ligeramente mejores en las métricas de fidelidad, lo que significa que las simplificaciones generadas se parecen más al texto de referencia. Por otro lado, el ruleset R2 mostró una ligera mejora en algunas métricas de legibilidad, lo que sugiere que puede ayudar a producir textos un poco más fáciles de leer.

Al analizar la combinación entre modelo y ruleset, la configuración que obtuvo el mejor equilibrio entre fidelidad y simplificación fue Llama 3 con el ruleset R0. Esta combinación logró los valores más altos en varias métricas importantes, lo que indica que produce simplificaciones más consistentes.

Otro resultado importante es que no se detectaron copias exactas del texto original. Esto significa que los modelos realmente están realizando una transformación del texto y no simplemente devolviendo la entrada original sin cambios.

También se observaron algunos casos problemáticos, principalmente en las salidas generadas por Mistral. En algunos ejemplos el modelo introdujo frases adicionales, cambios de tono o incluso partes en inglés, lo que indica que en ciertas situaciones puede desviarse del estilo esperado para la simplificación.

Finalmente, al revisar las métricas de legibilidad, se encontró un comportamiento interesante. Según el índice INFLESZ, los textos generados tienden a ser más fáciles de leer que los originales. Sin embargo, el índice Fernández-Huerta no muestra una mejora tan clara en promedio. Esto sugiere que diferentes métricas de legibilidad pueden capturar aspectos distintos del texto, por lo que es útil analizarlas de manera conjunta.

En general, los resultados de esta corrida permiten concluir que Llama 3 es actualmente el modelo más adecuado para continuar los experimentos, especialmente en combinación con el ruleset R0. Sin embargo, el ruleset R2 sigue siendo relevante para estudiar escenarios donde se busque una simplificación más orientada a mejorar la legibilidad.